# Chapter 11 — Build It Yourself: The Data Pipeline

Generate a synthetic story corpus, pack it into a memory-mappable `.bin`, then **implement**
`get_batch` — the random-window sampler that feeds every GPT training run — and watch a batch
decode back into real story text.

> 💡 **No GPU, no download.** It's all CPU + NumPy, on a corpus we build from scratch. Running a
> ✍️ cell before you fill it prints a friendly ⚠️, not an error.

In [ ]:
import numpy as np, torch, os, tempfile, random
DATA = tempfile.gettempdir()      # write the .bin shards here (works on Colab too)
print('ready — data dir:', DATA)

## Step 1 — Generate synthetic stories  ▶️ Run this

Real TinyStories was written by GPT using only simple words. We can't call an LLM, so we fill
templates from small word lists — the same idea: a simple, repetitive corpus a *tiny* model can
learn. Run it and read a couple of stories.

In [ ]:
NAMES = ['Lily','Tom','Max','Mia','Ben','Zoe','Sam','Ava','Leo','Ivy']
ANIMALS = ['cat','dog','frog','bird','bear','duck','fox','bunny']
ADJ = ['little','happy','small','brave','shy','kind','silly']
PLACES = ['park','forest','garden','pond','house','hill']
OBJECTS = ['ball','box','hat','key','leaf','shell','star']
VERBS = ['play','run','jump','sing','dance','hop']; EMO = ['happy','excited','proud','glad']

def make_story(rng):
    n,a,adj = rng.choice(NAMES), rng.choice(ANIMALS), rng.choice(ADJ)
    p,o,v,e = rng.choice(PLACES), rng.choice(OBJECTS), rng.choice(VERBS), rng.choice(EMO)
    return (f'Once upon a time, there was a {adj} {a} named {n}. {n} loved to {v} in the {p}. '
            f'One day, {n} found a {o}. {n} was very {e}. {n} and the {o} played all day. The end.\n')

rng = random.Random(0)
text = ''.join(make_story(rng) for _ in range(2000))
print(text[:190], '...')
print(f'\ncorpus: {len(text):,} characters from 2000 stories')

## Step 2 — Tokenize and pack to disk  ▶️ Run this

Tokenize once (char-level here), split off a validation slice, and write the ids to disk as raw
`uint16` — 2 bytes per token, no header. This is the `.bin` file training will memory-map.

In [ ]:
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
ids = np.array([stoi[c] for c in text], dtype=np.uint16)     # whole corpus as token ids

n = int(len(ids) * 0.9)                                       # 90/10 split, BEFORE sampling windows
ids[:n].tofile(os.path.join(DATA, 'ch11_train.bin'))
ids[n:].tofile(os.path.join(DATA, 'ch11_val.bin'))
print(f'vocab {len(chars)} | {n:,} train + {len(ids)-n:,} val tokens | {ids.nbytes/1e6:.2f} MB uint16')

## Step 3 — Implement `get_batch`  ✍️ Your turn

Memory-map the `.bin` and sample `BATCH` random windows. `x` is a window of `BLOCK` tokens; `y`
is the *same window shifted one token right* (the next-token targets). Cast to `int64` for the
embedding. Fill the two lines.

<details><summary>Stuck? Show the answer</summary>

```python
x = torch.stack([torch.from_numpy(data[i     : i + BLOCK    ].astype(np.int64)) for i in ix])
y = torch.stack([torch.from_numpy(data[i + 1 : i + 1 + BLOCK].astype(np.int64)) for i in ix])
```
</details>

In [ ]:
BLOCK, BATCH = 32, 8
def get_batch(split):
    data = np.memmap(os.path.join(DATA, f'ch11_{split}.bin'), dtype=np.uint16, mode='r')
    ix = torch.randint(len(data) - BLOCK, (BATCH,))       # random start offsets
    # ✍️ build x (windows of BLOCK tokens) and y (each shifted one right), as int64 tensors
    x = None      # replace
    y = None      # replace
    return x, y

torch.manual_seed(0)
xb, yb = get_batch('train')
print('x:', None if xb is None else tuple(xb.shape), '| y:', None if yb is None else tuple(yb.shape))

In [ ]:
# ▶️ Check your work
try:
    if xb is None or yb is None:
        print('⚠️  Fill in the ✍️ cell above (build x and y), then re-run. (Expected until you do.)')
    elif xb.shape != (BATCH, BLOCK) or yb.shape != (BATCH, BLOCK):
        print(f'⚠️  Wrong shape: x {tuple(xb.shape)}, y {tuple(yb.shape)}; expected ({BATCH}, {BLOCK}).')
    elif xb.dtype != torch.int64:
        print(f'⚠️  x is {xb.dtype} — cast to int64 (.astype(np.int64)) so it can index the embedding.')
    elif torch.equal(xb[:, 1:], yb[:, :-1]):
        print('✅ Correct — x and y are windows offset by one, int64, shape', tuple(xb.shape), '. That is a training batch.')
    else:
        print('⚠️  Shapes fine, but y is not x shifted by one — check the i vs i+1 offsets.')
except Exception as e:
    print('❌', e)

## Step 4 — Decode a batch back to text  ✍️ Your turn

A batch is just integers — but they *are* real story text. Build a `decode` (ids → string using
`itos`) and read `x[0]` and `y[0]`: you'll see `y` is `x` slid one character left.

<details><summary>Stuck? Show the answer</summary>

```python
decode = lambda t: ''.join(itos[int(i)] for i in t)
```
</details>

In [ ]:
# ✍️ build decode: a function turning a list/tensor of ids into a string, using itos
decode = None      # replace

if decode is not None and xb is not None:
    print('x[0]:', repr(decode(xb[0].tolist())))
    print('y[0]:', repr(decode(yb[0].tolist())))

In [ ]:
# ▶️ Check your work
try:
    if decode is None or xb is None:
        print('⚠️  Fill Step 3 (get_batch) and this cell (decode), then re-run. (Expected until you do.)')
    else:
        dx, dy = decode(xb[0].tolist()), decode(yb[0].tolist())
        if len(dx) == BLOCK and dx[1:] == dy[:-1]:
            print(f'✅ Real text! x[0] = {dx!r}')
            print(f'   y[0] = {dy!r}  (x shifted one char — the next-token targets).')
        else:
            print('⚠️  decode runs, but the shift check failed — revisit get_batch / decode.')
except Exception as e:
    print('❌', e)

## 🎉 You built a data pipeline.

Text → tokens → a memory-mappable `.bin` → random-window batches that decode back to real
stories. Point a Chapter 5 GPT's training loop at `get_batch` and it trains — the same machinery
scales from this toy corpus to trillions of tokens (only the `.bin` gets bigger).

Take it to the [mini-project](../project/), then on to
[Chapter 12 — KV-cache](../../12-inference-kv-cache/). 📚